# A2 - The Pipeline Tour: Real NLP in Three Lines

**Part A - What NLP Can Do | Notebook 2 of 3**

In A1 you did NLP by hand. You tokenized with TextBlob, pulled entities out of a spaCy
`doc`, and ran a topic model. It worked, but every task needed its own tool and its own
glue code. This notebook is the payoff. You will watch a pretrained transformer solve each
of those tasks (and several more) in about three lines, with no training at all.

By the end of this notebook you will be able to:

- Explain the `pipeline()` mental model: pick a task, get a default pretrained model, call it.
- Run sentiment analysis, zero-shot classification, named entity recognition, question
  answering, summarization, and fill-mask, each in a few lines.
- Use zero-shot classification to route text into your own custom categories with no training data.
- Swap models from the HuggingFace Hub with a single `model=` keyword and compare them.

**Prerequisites:** A1 (you have seen tokenization and NER done by hand). No PyTorch knowledge
needed yet. We treat the transformer as a black box on purpose. Part B opens that box.

**Runtime:** Google Colab. Every cell runs on CPU, so no GPU is required. A GPU is optional
and noticeably speeds up the heavier models (the summarization step in particular is slow on
CPU, since it generates text token by token). Each pipeline downloads its model on first use,
so the first call in a section may take a minute. Set the runtime with
`Runtime -> Change runtime type -> GPU` if you have one.

## Section 0: Environment Setup

We pin the HuggingFace stack to the `transformers` 4.x line on purpose. The 4.x line keeps
the beginner-friendly `summarization` and `question-answering` pipelines that this tour
relies on, and it installs alongside `numpy<2` (the version the rest of the course uses).
One caveat on this pinned stack: torch 2.2 will only load model weights in the modern
`safetensors` format, not the legacy `.bin` format, so for summarization we name a
`safetensors` model explicitly instead of relying on the older pipeline default.
Run the install cell once. On Colab you may be prompted to restart the runtime after the
install so the pinned versions are the ones actually imported.

In [ ]:
# Install the HuggingFace stack, pinned for this course.
# transformers 4.57.x is the last 4.x line: it keeps the summarization and
# question-answering pipelines that transformers 5.x removed, and it is happy with numpy<2.
# If Colab asks you to restart the runtime after this cell, do it, then continue.
!pip install -q "transformers==4.57.*" "datasets>=2.19,<4" "evaluate>=0.4.2" "accelerate>=0.30" "numpy<2"

# Download the small English spaCy pipeline (model weights, separate from pip).
!python -m spacy download en_core_web_sm

# TextBlob/NLTK tokenizer data (punkt_tab) for sentence/word tokenization.
import nltk
nltk.download('punkt_tab')


In [ ]:
# Standard imports for the whole notebook.
import numpy as np
import torch
from transformers import pipeline
import transformers

# Confirm we got the versions we pinned (transformers 4.57.x, numpy < 2).
print(f"transformers version: {transformers.__version__}")
print(f"numpy version:        {np.__version__}")

# Device selection. The pipeline() API uses an integer device argument:
#   device = 0   -> first CUDA GPU
#   device = -1  -> CPU
# We compute it once and pass it to every pipeline so GPU is used when available.
device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'GPU (cuda:0)' if device == 0 else 'CPU'}")

# Reproducibility. Pipelines are mostly deterministic for these tasks, but we set the seed
# so any sampling (for example in generation) behaves the same on every run.
SEED = 42
torch.manual_seed(SEED)

print("\nEnvironment setup complete.")

## What Are We Building Today?

You are an NLP engineer on a customer-support platform. Tickets land as raw text all day:

> "I have been charged twice for my subscription and support has not replied in 3 days.
> My account email is jordan@example.com and I am in Berlin."

The product team wants four things shipped this sprint: read the sentiment of each ticket,
route it to the right team, pull out the useful entities (people, places, emails), and
produce a one-line summary for the dashboard. There is no labelled training data and no
time to train models.

This is exactly the situation pretrained transformers were built for. In the rest of this
notebook we solve each of these tasks with the `pipeline()` API, no training required, then
swap in better models to see how easy it is to upgrade. By the end you will have a working
baseline for five or more NLP tasks and a clear sense of where these baselines are good
enough and where you would invest in fine-tuning later (Part C).

**Figure: the six NLP tasks one pipeline() call can solve**

![Six NLP tasks one pipeline call can solve](https://raw.githubusercontent.com/axel-sirota/ml_and_nlp/main/exercises/1-Pre-NLP-Tools/diagrams/pipeline-tour/six-task-tour.png)

In [ ]:
# One realistic support ticket we will reuse across several tasks, so you can see how the
# same input flows through different pipelines. Keep it short so every cell runs fast.
ticket = (
    "I have been charged twice for my subscription and support has not replied in 3 days. "
    "My account email is jordan@example.com and I am in Berlin."
)

print("Sample support ticket:\n")
print(ticket)

## The `pipeline()` Mental Model

Everything in this notebook is one idea repeated. The `pipeline()` function from
`transformers` takes a task name and hands you back a ready-to-call object:

```python
from transformers import pipeline
solver = pipeline("<task-name>")   # downloads a default pretrained model the first time
result = solver("some input text") # call it like a function
```

Three things happen behind that one line:

1. **Task to model.** HuggingFace looks up a sensible default pretrained model for that task
   and downloads it from the Hub (the first time only; after that it is cached and instant).
2. **Preprocess and run.** The text is tokenized into the numbers the model expects, run
   through the transformer, and the raw output is collected.
3. **Postprocess.** The numbers are turned back into something human: a label, a score, a
   span of text, a summary.

You do not have to know any of that yet. That is the whole point: the pipeline is a black
box. In Part B we build every piece inside it, and in Part C we fine-tune our own. For now,
learn the rhythm: **pick a task, get a model, call it.**

### Task 1: Sentiment Analysis (our first task)

We start with sentiment analysis: label a piece of text as positive or negative (with a
confidence score). The default model behind `pipeline("sentiment-analysis")` is
`distilbert-base-uncased-finetuned-sst-2-english`. Hold onto that name: it is
`distilbert-base-uncased` (a model we will fine-tune ourselves in Part C) already fine-tuned
on the SST-2 movie-review dataset. You are about to use the exact thing you will later build.

The output is a list of dicts, one per input:

```python
[{'label': 'POSITIVE', 'score': 0.9991}]
```

You can pass a single string or a list of strings. Passing a list runs them as a batch and
returns one result per input, which is how you would score a whole queue of tickets.

**Figure: what pipeline() does behind one line of code**

![What pipeline does behind one line of code](https://raw.githubusercontent.com/axel-sirota/ml_and_nlp/main/exercises/1-Pre-NLP-Tools/diagrams/pipeline-tour/pipeline-mental-model.png)

In [ ]:
# The smallest possible demo: one task, one call, one result.
# "sentiment-analysis" is a built-in task name. With no model= argument, HuggingFace picks
# a strong default and downloads it the first time (this first call may take a few seconds).
quick = pipeline("sentiment-analysis", device=device)

# Call it like a function. The result is a list with one dict per input.
print(quick("This three-line demo is honestly kind of magical."))
# Expect something like: [{'label': 'POSITIVE', 'score': 0.9998...}]

In [ ]:
# Build the sentiment pipeline once and reuse it.
classifier = pipeline("sentiment-analysis", device=device)

# A small batch of customer messages. Passing a list runs them together and returns
# one result dict per input, in order.
reviews = [
    "The new dashboard is fast and the support team fixed my issue in minutes.",
    "Billed me twice and nobody answered for three days. Cancelling.",
    "It works, I guess. Nothing special, nothing terrible.",
]

results = classifier(reviews)

# Each result is a dict with a 'label' (POSITIVE / NEGATIVE) and a 'score' (confidence 0..1).
for review, r in zip(reviews, results):
    print(f"[{r['label']}  {r['score']:.3f}]  {review}")

## Task 2: Zero-Shot Classification (No Training Data)

This is the one to remember. Sentiment analysis only knows the labels it was trained on
(positive / negative). But your support tickets need custom categories that no model was
trained on: `billing`, `technical`, `account`, `shipping`. Normally you would label
thousands of examples and train a classifier. Zero-shot classification skips all of that.

You give the pipeline your candidate labels at call time, in plain English, and it scores
how well each one fits, with no training data at all:

```python
router = pipeline("zero-shot-classification")
router("My card was charged twice", candidate_labels=["billing", "technical", "account"])
```

The default model is `facebook/bart-large-mnli`. The output is a single dict:

```python
{'sequence': 'My card was charged twice',
 'labels':  ['billing', 'account', 'technical'],   # sorted best-first
 'scores':  [0.95, 0.03, 0.02]}                     # aligned with labels
```

`labels[0]` is the prediction and `scores[0]` is its confidence. By default the scores sum
to 1 (the model picks one best label). Pass `multi_label=True` if more than one label can
apply at once. This is why zero-shot is a practitioner favourite: you can ship a router on
day one and change the categories by editing a Python list.

**Figure: how zero-shot classification routes a ticket**

![How zero-shot classification routes a ticket](https://raw.githubusercontent.com/axel-sirota/ml_and_nlp/main/exercises/1-Pre-NLP-Tools/diagrams/pipeline-tour/zero-shot-routing.png)

In [ ]:
# Build the zero-shot router once. The default model (facebook/bart-large-mnli) is larger,
# so the first download takes a bit longer; after that it is cached.
router = pipeline("zero-shot-classification", device=device)

# Our custom routing categories. These were never seen during the model's training:
# we are inventing them right here, in plain English.
candidate_labels = ["billing", "technical support", "account", "shipping"]

# Route our sample ticket. We pass the text plus the candidate labels.
result = router(ticket, candidate_labels=candidate_labels)

# labels and scores come back aligned and sorted best-first.
print(f"Ticket: {result['sequence']}\n")
for label, score in zip(result["labels"], result["scores"]):
    print(f"  {label:20s} {score:.3f}")

print(f"\nRoute this ticket to: {result['labels'][0].upper()}")

## Lab 1: Build Your Own Zero-Shot Ticket Router (15 min)

Your turn. You will route a small queue of support messages into your own categories using
zero-shot classification, with no training data.

**Steps:**

1. You already have a `router` pipeline from the demo above. Reuse it.
2. Define `my_labels`: a Python list of at least four routing categories that make sense for
   a support desk (think about the kinds of problems customers actually report).
3. For each message in `support_queue` (provided), call the router with your labels and keep
   the top predicted label.
4. Store the chosen labels in a list called `predicted_routes`, in the same order as the queue.

**Hints:**

- The router takes the text plus your candidate labels; check the Task 2 demo above for
  the exact keyword argument name.
- The result is sorted best-first, so the chosen route is the first entry of its label list.
- The verification cell below checks that you produced one route per message.

**Stretch (if you finish early):** add a confidence guardrail. If the top score is below
0.5, set the route to `"needs human review"` instead of the predicted label. This is exactly
how real routers avoid acting on low-confidence guesses.

In [ ]:
# Lab 1: Zero-shot ticket router. SOLUTION.

# A small queue of incoming support messages (provided).
support_queue = [
    "My payment failed but I was still charged. Please refund me.",
    "The app crashes every time I open the reports page.",
    "I need to change the email address on my account.",
    "Where is my order? It has been two weeks and nothing arrived.",
]

# 1. Define your own routing categories (at least four labels).
#    These are plain-English categories a support desk actually uses. The model never saw
#    them in training: that is the whole point of zero-shot. Edit this list and the router
#    re-routes with no retraining.
my_labels = ["billing", "technical issue", "account management", "shipping"]

# 2. Route each message and collect the top label for each one.
predicted_routes = []  # one route per message, same order as support_queue

for message in support_queue:
    # Call the router on the message with our candidate labels. The result dict has
    # "labels" sorted best-first, so result["labels"][0] is the predicted category.
    # Common mistake: passing my_labels positionally; it MUST be candidate_labels=.
    result = router(message, candidate_labels=my_labels)
    top_label = result["labels"][0]
    predicted_routes.append(top_label)

# --- Verification (provided) ---
if my_labels is not None and all(r is not None for r in predicted_routes):
    assert len(predicted_routes) == len(support_queue), "One route per message expected."
    print("Routing results:\n")
    for msg, route in zip(support_queue, predicted_routes):
        print(f"  -> {route:20s} | {msg}")
    print("\nNice. You built a working router with zero training data.")
else:
    print("Fill in my_labels and the routing loop, then re-run.")

# --- Stretch: confidence guardrail ---
# A real router should not act on a low-confidence guess. Here we re-route, but send
# anything below 0.5 to a human. result["scores"][0] is the top label's confidence.
print("\nStretch (with a 0.5 confidence guardrail):\n")
for message in support_queue:
    result = router(message, candidate_labels=my_labels)
    top_label, top_score = result["labels"][0], result["scores"][0]
    route = top_label if top_score >= 0.5 else "needs human review"
    print(f"  -> {route:20s} ({top_score:.2f}) | {message}")

## Task 3: Named Entity Recognition (Remember A1?)

In A1 you extracted entities by hand with spaCy: load `nlp`, run the text, read `doc.ents`.
Here is the same task as a one-line pipeline. The task name is `"ner"` (an alias of
`"token-classification"`), and we pass `aggregation_strategy="simple"`.

That aggregation argument matters. Transformers split words into subword tokens, so a name
like "jordan" might come back as `jo` + `##rdan`. With `aggregation_strategy="simple"`, the
pipeline stitches those pieces back into whole entities for you, returning a clean list:

```python
[{'entity_group': 'PER', 'score': 0.99, 'word': 'Jordan', 'start': 10, 'end': 16}, ...]
```

`entity_group` is the type (PER = person, LOC = location, ORG = organization), `word` is the
text, and `start`/`end` are character offsets. Same job spaCy did in A1, now from a
pretrained transformer in a single call.

In [ ]:
# Build the NER pipeline. aggregation_strategy="simple" merges subword pieces back into
# whole entities so we get clean, readable output.
ner = pipeline("ner", aggregation_strategy="simple", device=device)

# Run it on our support ticket (which mentions a person, a place, and an email).
entities = ner(ticket)

print(f"Ticket: {ticket}\n")
print("Entities found:")
for ent in entities:
    # entity_group: PER / LOC / ORG / MISC ; word: the matched text ; score: confidence
    print(f"  {ent['entity_group']:6s} {ent['word']:20s} (score {ent['score']:.3f})")

## Task 4: Question Answering

Extractive question answering takes a `context` (a passage) and a `question`, and returns
the span of the context that answers it. The default model is
`distilbert-base-cased-distilled-squad`, fine-tuned on the SQuAD dataset (the same dataset
the Part C Q&A chatbot uses).

You call it with two named arguments:

```python
qa(question="How long has support been silent?", context=ticket)
```

The output is a single dict:

```python
{'answer': '3 days', 'score': 0.87, 'start': 71, 'end': 77}
```

`answer` is the extracted text, `score` is the confidence, and `start`/`end` are the
character offsets inside the context. Note what this is NOT: it does not invent an answer,
it points at the part of the context that contains one.

In [ ]:
# Build the question-answering pipeline (default: distilbert fine-tuned on SQuAD).
qa = pipeline("question-answering", device=device)

# A slightly longer context so there is something to extract answers from.
context = (
    "Jordan opened a support ticket on Monday after being charged twice for the Pro plan. "
    "Support did not reply for 3 days. The refund of 49 dollars was finally issued on Friday."
)

# Ask a few questions against the same context.
questions = [
    "Who opened the support ticket?",
    "How long did support take to reply?",
    "How much was the refund?",
]

for q in questions:
    answer = qa(question=q, context=context)
    # answer['answer'] is the extracted span; answer['score'] is the confidence.
    print(f"Q: {q}\nA: {answer['answer']}  (score {answer['score']:.3f})\n")

## Task 5: Summarization

Summarization condenses a longer passage into a few sentences. Unlike the tasks above, this
one generates new text rather than picking a label or a span. We use
`facebook/bart-large-cnn`, a BART model fine-tuned for news summarization. We name it
explicitly because it ships `safetensors` weights, which load on the torch 2.2 /
transformers 4.57 stack this course pins; the older pipeline default ships only legacy
`.bin` weights that newer torch refuses to load for security reasons.

```python
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
summarizer(long_text, max_length=60, min_length=20)
```

The output is a list with one dict:

```python
[{'summary_text': 'A customer was charged twice and waited three days for a reply...'}]
```

Two gotchas worth knowing:

- `max_length` and `min_length` are measured in tokens and control the summary length.
- If your input is shorter than `max_length`, the model warns you (it cannot summarize a
  sentence into a longer summary). Feed it a real paragraph, not a single line.

In [ ]:
# Build the summarization pipeline.
# We name the model explicitly: facebook/bart-large-cnn ships safetensors weights,
# which load on torch 2.2 / transformers 4.57 without the torch>=2.6 .bin safe-load
# requirement that blocks the older distilbart-cnn-12-6 default. Same task, same API.
summarizer = pipeline("summarization", model="facebook/bart-large-cnn", device=device)

# A paragraph long enough to actually summarize. Too-short inputs trigger a length warning.
long_text = (
    "Over the past quarter our support platform handled a record number of tickets. "
    "Customers most often reported duplicate billing charges, slow response times, and "
    "trouble updating account details. The engineering team shipped a faster dashboard and "
    "an automated refund flow, which cut average resolution time from four days to under one. "
    "However, billing complaints remain the single largest category and are the focus for "
    "next quarter, along with proactive notifications when a payment is retried."
)

# Control the summary length in tokens. min_length avoids one-word summaries; max_length caps it.
summary = summarizer(long_text, max_length=60, min_length=20)

# Output is a list with one dict; the text lives under 'summary_text'.
print("Summary:\n")
print(summary[0]["summary_text"])

**Task 6: Fill-Mask (one more, for breadth).** Fill-mask is the actual pretraining task
behind models like BERT: write a sentence with a special mask token and the model predicts
the most likely words for the blank, ranked by probability. The default model is
`distilroberta-base` (mask token `<mask>`); BERT-family models use `[MASK]`, so read the
token off `unmasker.tokenizer.mask_token` rather than guessing. The output is a ranked list
of dicts like `[{'sequence': '... very frustrated ...', 'score': 0.21, 'token_str': ' frustrated'}, ...]`.

In [ ]:
# Task 6: Fill-mask. This is the actual pretraining task behind BERT-family models:
# predict the masked-out word. It is a nice peek at where these models come from.
# Default model: distilroberta-base, whose mask token is <mask>.
unmasker = pipeline("fill-mask", device=device)

# Read the correct mask token off the tokenizer instead of hard-coding it. The token differs
# by model family (<mask> for RoBERTa-family, [MASK] for BERT), so never guess it.
mask = unmasker.tokenizer.mask_token
print(f"This model's mask token is: {mask}\n")

# Predict the blank. Use the mask variable so this works regardless of the model.
sentence = f"The customer was very {mask} with the slow billing response."
predictions = unmasker(sentence)

# Output is a ranked list of dicts; show the top few candidate words and their scores.
for p in predictions[:5]:
    print(f"  {p['token_str']!r:15s} (score {p['score']:.3f})")

## Swapping Models From the HuggingFace Hub

Every pipeline so far used a default model. The real power of the Hub is that you can swap in
a different model for the same task with one keyword: `model=`.

```python
# Default sentiment model (POSITIVE / NEGATIVE):
pipeline("sentiment-analysis")

# A Twitter-tuned model that also knows NEUTRAL:
pipeline("sentiment-analysis", model="cardiffnlp/twitter-roberta-base-sentiment-latest")
```

Why swap?

- **Domain fit.** A model trained on tweets handles informal text better than one trained on
  movie reviews.
- **More classes.** Some sentiment models add a NEUTRAL label, which matters for support text.
- **Size and speed.** Distilled models (names often starting with `distil`) are roughly half
  the size and noticeably faster, trading a small amount of accuracy for latency.

How to choose? Open the model's page on huggingface.co and read its model card: what data it
was trained on, what labels it outputs, how big it is. Different models return different
label sets, so always check the output format after a swap. This "read the card, then swap"
habit is the practitioner's superpower for the rest of the course.

**Figure: same task, two models, different label sets**

![Same task, two models, different label sets](https://raw.githubusercontent.com/axel-sirota/ml_and_nlp/main/exercises/1-Pre-NLP-Tools/diagrams/pipeline-tour/model-swap.png)

## Lab 2: Swap a Model and Compare (Stretch, 12 min)

Now compare two models on the same task and see how the choice changes the answer.

**Steps:**

1. Build a second sentiment pipeline that uses
   `model="cardiffnlp/twitter-roberta-base-sentiment-latest"`. Call it `twitter_sentiment`.
2. You already have the default `classifier` from Task 1. Run BOTH models on every message
   in `tricky_messages` (provided), which includes sarcasm and neutral statements.
3. For each message, print the label and score from each model side by side.
4. Write a one-line observation (as a comment or print) about where the two models disagree.

**Hints:**

- Build the second pipeline exactly like the first, but add the `model=` argument.
- The Twitter model returns labels like `negative` / `neutral` / `positive`, while the
  default returns `POSITIVE` / `NEGATIVE`. That difference is the point: read each output.
- Each result is still a list with one dict, so the top result is `result[0]`.

**Homework extension (async, deeper):** turn this into a confidence-thresholded router. Run
the default sentiment model over a batch of messages, accept a label only when its score is
above a threshold you choose (say 0.9), and otherwise mark the message `"low confidence:
review"`. Then write two sentences on when you would stop using zero-shot or a generic
pretrained model and fine-tune your own instead. (Hint: think about narrow, high-stakes
categories like billing disputes. This is exactly the decision that leads into Part C.)

In [ ]:
# Lab 2: Swap a model and compare on the same task. SOLUTION.

# Messages that are genuinely hard: sarcasm, neutral tone, mixed feelings.
tricky_messages = [
    "Oh great, charged twice again. Exactly what I wanted today.",
    "The update installed without any problems.",
    "It is fine. Does the job. Would not rave about it.",
]

# 1. Build a second sentiment pipeline that uses the Twitter-tuned model.
#    Same call as the default classifier, but with the model= keyword pointing at a model
#    from the Hub. This one was trained on tweets and adds a NEUTRAL class. The first run
#    downloads it; after that it is cached.
twitter_sentiment = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    device=device,
)

# 2. Run BOTH models on each message and print their labels side by side.
#    You already have `classifier` (the default model) from Task 1.
if twitter_sentiment is not None:
    print(f"{'message':50s} | {'default':18s} | twitter-roberta")
    print("-" * 95)
    for msg in tricky_messages:
        # Each pipeline returns a list with one dict, so take result[0]. The two models use
        # different label vocabularies (POSITIVE/NEGATIVE vs negative/neutral/positive),
        # which is exactly why we read each output instead of assuming a fixed set.
        default_pred = classifier(msg)[0]
        twitter_pred = twitter_sentiment(msg)[0]
        if default_pred is not None and twitter_pred is not None:
            d = f"{default_pred['label']} {default_pred['score']:.2f}"
            t = f"{twitter_pred['label']} {twitter_pred['score']:.2f}"
            print(f"{msg[:48]:50s} | {d:18s} | {t}")
else:
    print("Build twitter_sentiment first, then fill in the comparison loop.")

# 3. Where the two models disagree, and why.
# The default model only has POSITIVE/NEGATIVE, so it is forced to commit on the neutral and
# sarcastic lines: it tends to call "It is fine. Does the job." POSITIVE and may miss the
# sarcasm in "Oh great, charged twice again." The Twitter model has a NEUTRAL class and was
# trained on informal text, so it usually tags the lukewarm line neutral and is better at
# the sarcasm. The lesson: the label set and training domain change the answer, so always
# read the model card and check the output format after a swap.

In [ ]:
# A glimpse of where this course is heading. Gradio can wrap any pipeline in a UI in one
# line with gr.Interface.from_pipeline. We guard the import so this notebook still runs
# end-to-end in an environment without gradio installed.
try:
    import gradio as gr

    # Turn our sentiment classifier into an interactive web UI. In Colab this launches an
    # inline app you can type into. The pipeline IS the backend; Gradio just wraps it.
    demo = gr.Interface.from_pipeline(classifier)
    # demo.launch()  # uncomment in Colab to open the interactive app
    print("Gradio is available. Uncomment demo.launch() to try the interactive app.")
    print("This is the shape of the final deliverable: a model behind a simple chat UI.")
except ImportError:
    # No gradio in this environment. The point still stands: a pipeline is a ready backend.
    print("gradio is not installed here (that is fine).")
    print("Key idea: gr.Interface.from_pipeline(classifier) would wrap this pipeline in a")
    print("web UI in one line. In Part C you will load YOUR fine-tuned model the same way.")

## What You Learned, and Where This Goes Next

In one notebook, with no training, you solved six NLP tasks:

| Task | Pipeline | Model |
|------|----------|-------|
| Sentiment | `pipeline("sentiment-analysis")` | distilbert-base-uncased-finetuned-sst-2 (default) |
| Zero-shot routing | `pipeline("zero-shot-classification")` | facebook/bart-large-mnli (default) |
| Named entities | `pipeline("ner", aggregation_strategy="simple")` | bert-large finetuned conll03 (default) |
| Question answering | `pipeline("question-answering")` | distilbert-base-cased-distilled-squad (default) |
| Summarization | `pipeline("summarization", model="facebook/bart-large-cnn")` | facebook/bart-large-cnn |
| Fill-mask | `pipeline("fill-mask")` | distilroberta-base (default) |

**Key takeaways:**

- The pattern never changed: pick a task, get a pretrained model, call it.
- Zero-shot classification gives you a custom classifier with no labelled data: edit a list,
  re-route. It is your fastest path to a working baseline.
- Swap models from the Hub with one `model=` keyword; always read the model card and check
  the output format, because labels differ between models.
- Pretrained baselines are excellent starting points, but for narrow, high-stakes categories
  you will eventually fine-tune your own model. That is Part C.

**Production take-homes (worth remembering):**

- The first call downloads and caches the model; later calls are instant.
- Pin a model `revision="<commit-hash>"` when you need reproducible results across runs.
- Pass a list and set `batch_size=` to score many inputs efficiently on a GPU.

**The bridge.** Every pipeline here was a sealed black box. Inside it, three things happened:
a tokenizer turned text into numbers, a transformer (built from PyTorch tensors and neural
network layers) processed them, and a postprocessor turned the output back into a label or a
span. In **Part B** you build every one of those pieces yourself: tensors, word embeddings,
neural networks, and an MLP classifier. In **Part C** you fine-tune `distilbert-base-uncased`
(the very model behind today's sentiment pipeline) on your own data and load it into a Gradio
chatbot like the preview above. You have seen the destination. Now we learn how to get there.

**Next notebook:** A3 - Capstone A, where you ship a real customer-support router on the
Twitter support dataset using the zero-shot skill you just learned.

**Figure: how today's black box opens across Parts B and C**

![How today's black box opens across Parts B and C](https://raw.githubusercontent.com/axel-sirota/ml_and_nlp/main/exercises/1-Pre-NLP-Tools/diagrams/pipeline-tour/black-box-bridge.png)